In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO

In [2]:
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")
    
logging.getLogger().setLevel(logging.ERROR)
    
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

In [3]:
load_dotenv()
rag_directory = os.getenv('DIRECTORY', 'Text_Document')

In [4]:
def load_documents(directory):
    loader = DirectoryLoader(directory)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", "។", "?", "!", ""]
    )
    docs = text_splitter.split_documents(documents)
    return docs

In [5]:
load_documents(rag_directory)

[Document(metadata={'source': 'Text_Document\\History.txt'}, page_content='..ជំពូកទី 1 ប្រវត្តិសាស្ត្រពិភពលោក\n\nសង្គ្រាមលោកលើកទី 2 ត្រូវបានបញ្ចប់នៅឆ្នាំ 1945 ។ សង្គ្រាមថ្មីមួយទៀតបានកើតឡើងគឺ សង្គ្រាមត្រជាក់(សង្គ្រាមមនោគមន៍វិជ្ជា) ដែលបង្កើតឡើងដោយមហាអំណាចធំៗពីរលើពិភពលោក គឺសហភាពសូវៀត និងសហរដ្ឋអាមេរិក ។ សង្គ្រាមនេះបានអូសបន្លាយអស់រយៈពេលជាង 40 ឆ្នាំ (1945-1990) ។ សង្គ្រាមនេះបញ្ចប់ត្រឹមលោកប្រធានាធិបតីសហរដ្ឋអាមេរិក រ៉ូណាលវីគេន និងប្រធា នាធិបតីសហភាពសូវៀត មីខាអ៊ុល ហ្គោបាឆ្លូវ ជួបចរចាគ្នានិងការវាយបំបែករបងជញ្ជាំងក្រងបែរ ឡាំង ។ សង្គ្រាមត្រជាក់បានបញ្ចប់ តែសង្គ្រាមភេរវកម្មបានកើតឡើងរំខានសេចក្តីសុខសាន្ត និងការ អភិវឌ្ឍរបស់ពិភពលោក ។\n\n+.មេរៀនទី 1 សហរដ្ឋអាមេរិក (1860-1990)\n\nសហរដ្ឋអាមេរិកជាទឹកដីនៃជាតិឥណ្ឌាស្បែកក្រហម ។ នៅដើមសតវត្សទី 16 មានពួកនិគមជន អឺរ៉ុបបានទៅតាំងលំនៅ ដោយបានបំផ្លាញភូមិករ និងជីវិតជនជាតិដើម។ សហរដ្ឋអាមេរិកក្លាយជា ទឹកដីនៃជនភៀសខ្លួនពីទ្វីបអឺរ៉ុបម្តងបន្តិចៗរហូតដល់បង្កើតជារដ្ឋ 13 នៅត្រើយអាត្លង់ទិកប៉ែកខាង កើត ។ សហរដ្ឋអាមេរិកមានផ្ទៃក្រឡា 9 363 000 km² ឈរលើលំដាប់ថ្នាក់ទី 4 បន្ទាប់ពីសហព័ន្ធរុស្ស៊ី 

In [6]:
len(load_documents(rag_directory))

411

In [7]:
class E5Embeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

In [8]:
def main():
    docs = load_documents(rag_directory)

    with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
        embedding = E5Embeddings(
            model_name="intfloat/multilingual-e5-base",
            model_kwargs={"device": "cpu"},        # "cuda" if available
            encode_kwargs={"normalize_embeddings": True}
        )

        Chroma.from_documents(
            documents=docs,
            embedding=embedding,    
            persist_directory="./chroma_db",
            collection_name="khmer_corpus",
            collection_metadata={"hnsw:space": "cosine"}  # cosine for normalized vectors
        )

if __name__ == "__main__":
    main()
